# Segment anomaly detection

## The task
Segment anomaly detection is the task of identifying segments of a time series where the data behaves differently than expected.
The goal is to estimate starts and ends of such segments.
It is an important special case of change detection where certain segments are deemed "normal" and others are "anomalous". In most settings, a vast majority of the data is "normal".

We use the same data as in the [change detection intro](../change_detection/intro.ipynb), but now we consider the segments `100:140` and `220:300` as segment anomalies, and the remaining data as "normal" or "baseline" data.

In [ ]:
from skchange.datasets import generate_piecewise_normal_data

x = generate_piecewise_normal_data(
    means=[0, [8.0, 0.0, 0.0], 0.0, [2.0, 3.0, 5.0]],
    lengths=[100, 40, 80, 80],
    seed=8,
)
x.columns = ["var0", "var1", "var2"]
x.index.name = "time"
x

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook"

px.line(x)

As for change detection, segment anomalies may also affect the data in numerous other ways than sudden jumps in the mean.

### Composable segment anomaly detectors
Let us use the `CAPA` detector to detect segment anomalies in the toy data. It consists of the same components as the change detector we used before: A detector (`CAPA`), an interval score (`segment_saving`) and a penalty (`segment_penalty`). "Savings" is one of two types of anomaly scores supported in Skchange. You can read more about them in the [Concepts](./concepts/interval_scores.ipynb) section.

In [ ]:
from skchange.anomaly_detectors import CAPA
from skchange.anomaly_scores import L2Saving

detector = CAPA(
    segment_saving=L2Saving(),
    segment_penalty=20,
)
detector

### `fit`
We fit the detector to obtain a fitted instance.

In [ ]:
detector.fit(x)

### `predict`
As for change detection, `predict` is used to detect segment anomalies in test data `x`. The output is a `pd.DataFrame` with the `"ilocs"` column holding the integer locations of segment anomalies as `pd.Interval`s, and the `"labels"` column holding unique labels for each segment. The labels run from 1, ..., K, where K is the number of detected segment anomalies.

In [ ]:
detections = detector.predict(x)
detections

In [ ]:
from skchange.utils.plotting import plot_detections

plot_detections(x, detections, data_repr="line")

### `transform`
The `transform` method labels the data according to the segment anomaly segmentation. The output is a `pd.DataFrame` with the same index as the input `x` and an integer column `"labels"` indicating which segment the index belongs to. The label `0` denotes the normal segments, and the labels `>0` denote the segment anomalies.

In [ ]:
labels = detector.transform(x)
labels

In [ ]:
px.line(labels)

### `transform_scores`
`CAPA` also supportes the `transform_scores` method. It returns the cumulative optimal penalised saving at each index.

In [ ]:
capa_scores = detector.transform_scores(x)
px.line(capa_scores)